In [ ]:
# ============================================================
# ERA5 preprocessing
# 1994-2014
#
# Output variables:
#   tas      : K
#   sfcWind  : m s-1
#   rsds     : W m-2
# ============================================================

import xarray as xr
import numpy as np
from pathlib import Path
import sys


# --- centralized path bootstrap (see config/project_paths.json) ---
_PROJECT_ROOT = Path.cwd()
while not (_PROJECT_ROOT / "scripts" / "common" / "project_paths.py").exists():
    _PROJECT_ROOT = _PROJECT_ROOT.parent
sys.path.insert(0, str(_PROJECT_ROOT / "scripts" / "common"))
from project_paths import get_path
# -------------------------------------------------------------------


# ============================================================
# 1. File paths
# ============================================================

ERA5_FILE = get_path("data_raw_era5_raw") / "116b0ea5b6757707487efaa87f9aaae.grib"

OUTPUT_FILE = get_path("data_raw_era5_raw") / "ERA5_1994_2014_wind_solar.nc"


# ============================================================
# 2. Check file
# ============================================================

if not ERA5_FILE.exists():
    raise FileNotFoundError(
        f"ERA5 file not found:\n{ERA5_FILE}"
    )


# ============================================================
# 3. Read u10, v10, t2m
# ============================================================

print("=" * 60)
print("Reading u10, v10 and t2m")
print("=" * 60)

ds = xr.open_dataset(
    ERA5_FILE,
    engine="cfgrib"
)

print(ds)


# ============================================================
# 4. Read ssrd separately
# ============================================================

print("\n" + "=" * 60)
print("Reading ssrd")
print("=" * 60)

ds_ssrd = xr.open_dataset(
    ERA5_FILE,
    engine="cfgrib",
    backend_kwargs={
        "filter_by_keys": {
            "shortName": "ssrd"
        }
    }
)

print(ds_ssrd)


# ============================================================
# 5. Calculate wind speed
# ============================================================

u10 = ds["u10"]
v10 = ds["v10"]

sfcWind = np.sqrt(
    u10**2 + v10**2
)

sfcWind.name = "sfcWind"

sfcWind.attrs = {
    "long_name": "10 m wind speed",
    "standard_name": "wind_speed",
    "units": "m s-1"
}


# ============================================================
# 6. Temperature
#
# Keep K!
# CMIP6 tas is normally also K.
# ============================================================

tas = ds["t2m"]

tas.name = "tas"

tas.attrs = {
    "long_name": "2 m air temperature",
    "standard_name": "air_temperature",
    "units": "K"
}


# ============================================================
# 7. Convert ssrd to rsds
# ============================================================

ssrd = ds_ssrd["ssrd"]

print("\nRaw ssrd:")
print(
    "min  =",
    ssrd.min().item(),
    "J/m²"
)

print(
    "mean =",
    ssrd.mean().item(),
    "J/m²"
)

print(
    "max  =",
    ssrd.max().item(),
    "J/m²"
)


rsds = ssrd / 86400.0

rsds.name = "rsds"

rsds.attrs = {
    "long_name":
        "Surface downwelling shortwave radiation",

    "standard_name":
        "surface_downwelling_shortwave_flux_in_air",

    "units":
        "W m-2"
}


# ============================================================
# 8. Align ssrd time with the main ERA5 time
# ============================================================

print("\nMain time:")
print(ds.time.values[:3])
print(ds.time.values[-3:])

print("\nssrd valid time:")
print(ds_ssrd.valid_time.values[:3])
print(ds_ssrd.valid_time.values[-3:])


rsds = rsds.assign_coords(
    time=ds.time.values
)


# ============================================================
# 9. Remove GRIB auxiliary coordinates
# ============================================================

def clean_grib_coords(da):

    for coord in [
        "valid_time",
        "number",
        "step",
        "surface"
    ]:

        if coord in da.coords:
            da = da.drop_vars(coord)

    return da


tas = clean_grib_coords(tas)
sfcWind = clean_grib_coords(sfcWind)
rsds = clean_grib_coords(rsds)


# ============================================================
# 10. Make time coordinates identical
# ============================================================

common_time = ds.time.values

tas = tas.assign_coords(
    time=common_time
)

sfcWind = sfcWind.assign_coords(
    time=common_time
)

rsds = rsds.assign_coords(
    time=common_time
)


# ============================================================
# 11. Create Dataset
# ============================================================

era5 = xr.Dataset({

    "tas": tas,

    "sfcWind": sfcWind,

    "rsds": rsds

})


# ============================================================
# 12. Sort
# ============================================================

era5 = era5.sortby("time")


# ============================================================
# 13. Dataset attributes
# ============================================================

era5.attrs = {

    "title":
        "ERA5 monthly wind, temperature and solar radiation",

    "period":
        "1994-2014",

    "spatial_resolution":
        "0.25 degree",

    "source":
        "ERA5 reanalysis",

    "variables":
        "tas, sfcWind, rsds"
}


# ============================================================
# 14. Basic checks
# ============================================================

print("\n" + "=" * 60)
print("FINAL DATASET")
print("=" * 60)

print(era5)


print("\nDimensions:")

print(
    "time     =",
    era5.sizes["time"]
)

print(
    "latitude =",
    era5.sizes["latitude"]
)

print(
    "longitude =",
    era5.sizes["longitude"]
)


# ============================================================
# 15. Check time
# ============================================================

assert era5.sizes["time"] == 252

print("\nTime:")
print("First:", era5.time.values[0])
print("Last :", era5.time.values[-1])


# ============================================================
# 16. Check variables
# ============================================================

print("\nVariables:")

for var in era5.data_vars:

    print(
        f"{var:10s}",
        "shape =", era5[var].shape,
        "units =", era5[var].attrs.get("units")
    )


# ============================================================
# 17. Physical checks
# ============================================================

print("\n" + "=" * 60)
print("Physical checks")
print("=" * 60)


print("\nTemperature:")
print(
    "min  =",
    era5.tas.min().item(),
    "K"
)

print(
    "mean =",
    era5.tas.mean().item(),
    "K"
)

print(
    "max  =",
    era5.tas.max().item(),
    "K"
)


print("\nWind speed:")
print(
    "min  =",
    era5.sfcWind.min().item(),
    "m/s"
)

print(
    "mean =",
    era5.sfcWind.mean().item(),
    "m/s"
)

print(
    "max  =",
    era5.sfcWind.max().item(),
    "m/s"
)


print("\nSolar radiation:")
print(
    "min  =",
    era5.rsds.min().item(),
    "W/m²"
)

print(
    "mean =",
    era5.rsds.mean().item(),
    "W/m²"
)

print(
    "max  =",
    era5.rsds.max().item(),
    "W/m²"
)


# ============================================================
# 18. Missing values
# ============================================================

print("\n" + "=" * 60)
print("Missing values")
print("=" * 60)

for var in era5.data_vars:

    missing = era5[var].isnull().sum().item()

    total = era5[var].size

    percentage = (
        missing / total * 100
    )

    print(
        f"{var:10s}: "
        f"{missing:,} "
        f"({percentage:.4f}%)"
    )


# ============================================================
# 19. Save NetCDF
# ============================================================

print("\n" + "=" * 60)
print("Saving...")
print("=" * 60)


encoding = {

    "tas": {
        "zlib": True,
        "complevel": 4,
        "dtype": "float32"
    },

    "sfcWind": {
        "zlib": True,
        "complevel": 4,
        "dtype": "float32"
    },

    "rsds": {
        "zlib": True,
        "complevel": 4,
        "dtype": "float32"
    }
}


era5.to_netcdf(
    OUTPUT_FILE,
    encoding=encoding
)


# ============================================================
# 20. Final
# ============================================================

print("\n" + "=" * 60)
print("SUCCESS")
print("=" * 60)

print(
    "Output:",
    OUTPUT_FILE
)
